In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

## Ingestion del carpeta "movie_company"

###Paso 1 - Leer los archivos CSV usando "DataframeReader" de Spark

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType

In [0]:
movies_companies_schema = StructType([
    StructField("movieId", IntegerType(), False),
    StructField("companyId", IntegerType(), False)
])

In [0]:
movies_companies_df = spark.read\
    .schema(movies_companies_schema)\
    .csv(f"{bronze_folder_path}/{v_file_date}/movie_company")

### Paso 2 - Renombrar las columnas y añadir nuevas columnas

In [0]:
from pyspark.sql.functions import lit, current_timestamp

In [0]:
movies_companies_final_df = add_ingestion_date(movies_companies_df)\
    .withColumnsRenamed({"companyId": "company_id",
                         "movieId": "movie_id"})\
    .withColumn("environment", lit("Production"))\
    .withColumn("file_date", lit(v_file_date))

### Paso 3 - Escribir la salida en un formato "Parquet" PartitionBy

In [0]:
#overwrite_partition("movie_silver", "movies_companies", "file_date", v_file_date)

In [0]:
#movies_companies_final_df.write.mode("overwrite").parquet(f"{silver_folder_path}/movies_companies")

In [0]:
#movies_companies_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.movies_companies")

condition_merge = 'tgt.movie_id = src.movie_id AND tgt.company_id = src.company_id AND tgt.file_date = src.file_date'

incremental_merge("movie_silver", "movies_companies", movies_companies_final_df, condition_merge, "file_date")

In [0]:
%sql
SELECT file_date, count(1)
FROM movie_silver.movies_companies
GROUP BY file_date;

file_date,count(1)
2024-12-16,8000
2024-12-23,4000
2024-12-30,1677


In [0]:
dbutils.notebook.exit("Exitoso")